# Experiment 7: Dimensionality Reduction and Model Evaluation (With and Without PCA)
This standalone notebook applies Principal Component Analysis (PCA) with 95% variance retention, evaluating 10 classifiers with and without PCA using 5-Fold Cross Validation and statistical tests.

In [1]:
import os
import matplotlib
matplotlib.use('Agg') # Strictly headless - non-interfering, zero GUI popups

def resolve_path(rel_path):
    """Dynamically resolves datasets whether run from repo root or Ex subfolder."""
    for prefix in ['', '../', '../../']:
        cand = os.path.join(prefix, rel_path)
        if os.path.exists(cand):
            return cand
    return rel_path

def resolve_out(rel_path):
    """Avoids nested directories if running from within Ex7."""
    if os.path.basename(os.getcwd()) == 'Ex7':
        if rel_path.startswith('Ex7/'):
            return rel_path[len('Ex7/'):]
    return rel_path

import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score
from scipy.stats import ttest_rel, friedmanchisquare
import matplotlib.pyplot as plt
import seaborn as sns

os.makedirs(resolve_out('Ex7'), exist_ok=True)

In [2]:
def run_pca_analysis(X_scaled):
    pca = PCA().fit(X_scaled)
    cum_var = np.cumsum(pca.explained_variance_ratio_)
    n_components = np.argmax(cum_var >= 0.95) + 1
    
    plt.figure(figsize=(8, 5))
    plt.plot(range(1, len(cum_var) + 1), cum_var, marker='o', linestyle='--')
    plt.axhline(y=0.95, color='r', linestyle='-')
    plt.axvline(x=n_components, color='r', linestyle='-')
    plt.title('Scree Plot: Explained Variance by Components')
    plt.xlabel('Number of Components')
    plt.ylabel('Cumulative Explained Variance')
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(resolve_out('Ex7/Scree_Plot.eps'), format='eps', dpi=300)
    plt.close()
    
    pca_final = PCA(n_components=n_components)
    X_pca = pca_final.fit_transform(X_scaled)
    return X_pca, n_components

def evaluate_models(X_tr, y_tr, X_te, y_te, X_full, y_full):
    models = {
        'SVM': SVC(C=1.0, kernel='linear', random_state=42),
        'Naive Bayes': GaussianNB(),
        'KNN': KNeighborsClassifier(n_neighbors=5),
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
        'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
        'Random Forest': RandomForestClassifier(n_estimators=50, random_state=42),
        'AdaBoost': AdaBoostClassifier(n_estimators=50, random_state=42),
        'Gradient Boosting': GradientBoostingClassifier(n_estimators=50, learning_rate=0.1, random_state=42)
    }
    results = {}
    cv_scores = {}
    for name, model in models.items():
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_te)
        results[name] = {
            'Accuracy': accuracy_score(y_te, y_pred),
            'F1': f1_score(y_te, y_pred)
        }
        cv_scores[name] = cross_val_score(model, X_full, y_full, cv=5, scoring='accuracy')
    return results, cv_scores

In [3]:
def run_experiment_7(csv_path="Datasets/Breast_Cancer/breast_cancer.csv"):
    print("="*60)
    print("=== LAUNCHING EXPERIMENT 7: PCA & MODEL EVALUATION ===")
    print("="*60)
    
    path = resolve_path(csv_path)
    df = pd.read_csv(path)
    if 'id' in df.columns:
        df = df.drop(columns=['id'])
    if 'Unnamed: 32' in df.columns:
        df = df.drop(columns=['Unnamed: 32'])
        
    target_col = 'diagnosis' if 'diagnosis' in df.columns else df.columns[0]
    X = df.drop(columns=[target_col])
    y = df[target_col].map({'M': 1, 'B': 0}) if df[target_col].dtype == 'object' else df[target_col]
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # No-PCA
    X_tr_no, X_te_no, y_tr, y_te = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)
    res_no, cv_no = evaluate_models(X_tr_no, y_tr, X_te_no, y_te, X_scaled, y)
    
    # With-PCA
    X_pca, n_comp = run_pca_analysis(X_scaled)
    X_tr_pca, X_te_pca, _, _ = train_test_split(X_pca, y, test_size=0.2, random_state=42, stratify=y)
    res_pca, cv_pca = evaluate_models(X_tr_pca, y_tr, X_te_pca, y_te, X_pca, y)
    
    print("\n=== EXPERIMENT 7 PIPELINE COMPLETE ===")
    return {'res_no': res_no, 'res_pca': res_pca, 'cv_no': cv_no, 'cv_pca': cv_pca, 'n_comp': n_comp}

In [4]:
# Master Execution Cell
ex7_output = run_experiment_7()

n_orig, n_comp = 30, ex7_output['n_comp']
df_no = pd.DataFrame(ex7_output['res_no']).T
df_pca = pd.DataFrame(ex7_output['res_pca']).T
df_res = pd.DataFrame({
    f'Acc (No-PCA, {n_orig} feats)': df_no['Accuracy'],
    f'Acc (PCA, {n_comp} feats)': df_pca['Accuracy'],
    'Acc Delta': df_pca['Accuracy'] - df_no['Accuracy'],
    f'F1 (No-PCA, {n_orig} feats)': df_no['F1'],
    f'F1 (PCA, {n_comp} feats)': df_pca['F1'],
    'F1 Delta': df_pca['F1'] - df_no['F1']
})
display(df_res.style.format("{:.4f}").background_gradient(cmap='RdYlGn', subset=['Acc Delta', 'F1 Delta']))

=== LAUNCHING EXPERIMENT 7: PCA & MODEL EVALUATION ===



=== EXPERIMENT 7 PIPELINE COMPLETE ===


,"Acc (No-PCA, 30 feats)","Acc (PCA, 10 feats)",Acc Delta,"F1 (No-PCA, 30 feats)","F1 (PCA, 10 feats)",F1 Delta
SVM,0.9737,0.9561,-0.0175,0.9790,0.9645,-0.0145
Naive Bayes,0.9298,0.9298,0.0000,0.9444,0.9444,0.0000
KNN,0.9649,0.9561,-0.0088,0.9726,0.9655,-0.0071
Logistic Regression,0.9825,0.9649,-0.0175,0.9861,0.9718,-0.0143
Decision Tree,0.9211,0.9123,-0.0088,0.9362,0.9296,-0.0066
Random Forest,0.9561,0.9474,-0.0088,0.9655,0.9583,-0.0072
AdaBoost,0.9561,0.9561,0.0000,0.9660,0.9660,0.0000
Gradient Boosting,0.9474,0.9211,-0.0263,0.9589,0.9371,-0.0218
